In [ ]:
!pip install diffusers transformers accelerate controlnet-aux
!pip install git+https://github.com/openai/CLIP.git
!pip install segment-anything

In [ ]:
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
from diffusers import UniPCMultistepScheduler
from transformers import pipeline as hf_pipeline
from PIL import Image
import requests
from io import BytesIO
import torch
import numpy as np

print("All imports done!")

In [ ]:
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-depth",
    torch_dtype=torch.float16
)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16
).to("cuda")

pipe.scheduler = UniPCMultistepScheduler.from_config(
    pipe.scheduler.config
)
pipe.enable_model_cpu_offload()
pipe.enable_attention_slicing()

print("Pipeline loaded!")

In [ ]:
depth_estimator = hf_pipeline(
    "depth-estimation",
    model="LiheYoung/depth-anything-base-hf"
)

print("Depth estimator loaded!")

In [ ]:
!pip install diffusers transformers accelerate controlnet-aux xformers

In [ ]:
url = "https://images.unsplash.com/photo-1555041469-a586c61ea9bc?w=512"
response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
image = Image.open(BytesIO(response.content)).convert("RGB")
image = image.resize((512, 512))

print("Image loaded!")
image

In [ ]:
depth_result = depth_estimator(image)
depth_map = depth_result["depth"].resize((512, 512))

print("Depth map ready!")
depth_map

In [ ]:
result = pipe(
    prompt="a modern scandinavian living room, \
            warm lighting, minimalist furniture, \
            hardwood floors, photorealistic, \
            interior design magazine, 8k",
    
    negative_prompt="ugly, blurry, bad proportions, \
                     unrealistic, cartoon, drawing, \
                     dark, gloomy",
    
    image=depth_map,
    num_inference_steps=20,
    guidance_scale=7.5
).images[0]

print("Generation complete!")
result

In [ ]:
styles = [
    "a modern scandinavian living room, warm lighting, minimalist, photorealistic",
    "a luxurious royal living room, gold accents, velvet furniture, photorealistic",
    "an industrial loft, exposed brick walls, metal fixtures, photorealistic",
    "a bohemian cozy room, warm earth tones, plants everywhere, photorealistic"
]

results = []

for style in styles:
    output = pipe(
        prompt=style,
        negative_prompt="ugly, blurry, unrealistic, cartoon",
        image=depth_map,
        num_inference_steps=20,
        guidance_scale=7.5
    ).images[0]
    results.append(output)
    print(f"Generated: {style[:40]}...")

print("All styles generated!")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
style_names = ["Scandinavian", "Royal Luxury", "Industrial", "Bohemian"]

for idx, (ax, result, name) in enumerate(zip(axes.flatten(), results, style_names)):
    ax.imshow(result)
    ax.set_title(name, fontsize=14)
    ax.axis("off")

plt.tight_layout()
plt.savefig("style_comparison.png")
plt.show()
print("Saved!")

In [ ]:
print(pipe)       # Should show StableDiffusionControlNetPipeline
print(depth_map)  # Should show PIL Image info
print(image)      # Should show PIL Image info

In [ ]:
!pip install segment-anything
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

from segment_anything import SamPredictor, sam_model_registry
import numpy as np

sam = sam_model_registry["vit_b"](checkpoint="sam_vit_b_01ec64.pth")
sam.to("cuda")
predictor = SamPredictor(sam)

print("SAM loaded!")

In [ ]:
import numpy as np

# Convert image to numpy array
image_np = np.array(image)

# MUST set image before predicting
predictor.set_image(image_np)

# Multiple click points to cover full sofa
click_points = np.array([
    [256, 250],
    [256, 300],
    [256, 350],
])
click_labels = np.array([1, 1, 1])

masks, scores, logits = predictor.predict(
    point_coords=click_points,
    point_labels=click_labels,
    multimask_output=False
)

print(f"Mask generated! Confidence: {scores[0]:.3f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Original image
axes[0].imshow(image)
axes[0].set_title("Original Image")
axes[0].axis("off")

# Image with mask overlay
axes[1].imshow(image)
axes[1].imshow(masks[0], alpha=0.5, cmap="cool")
axes[1].plot(256, 300, "r*", markersize=15)  # Show click point
axes[1].set_title("Segmented Object")
axes[1].axis("off")

plt.tight_layout()
plt.savefig("sam_segmentation.png")
plt.show()
print("Segmentation visualized!")

In [ ]:
from PIL import Image, ImageFilter
import numpy as np

mask_image = Image.fromarray((masks[0] * 255).astype(np.uint8))
mask_blurred = mask_image.filter(ImageFilter.GaussianBlur(radius=3))

depth_result = depth_estimator(image)
depth_map = depth_result["depth"].resize((512, 512))

result_masked = pipe(
    prompt="a clean modern white fabric sofa, \
            scandinavian style, wooden legs, \
            photorealistic, interior design magazine",
    negative_prompt="ugly, blurry, unrealistic, \
                     cartoon, gold, gaudy, shiny",
    image=depth_map,
    num_inference_steps=20,
    guidance_scale=7.5
).images[0]

result_composite = Image.composite(result_masked, image, mask_blurred)
print("Done!")
result_composite

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(image)
axes[0].set_title("Original Room")
axes[0].axis("off")

axes[1].imshow(masks[0], cmap="cool")
axes[1].set_title("Selected Object (SAM Mask)")
axes[1].axis("off")

axes[2].imshow(result_composite)
axes[2].set_title("Redesigned Object Only")
axes[2].axis("off")

plt.tight_layout()
plt.savefig("object_redesign.png")
plt.show()
print("Saved!")

In [ ]:
!pip install git+https://github.com/openai/CLIP.git

import clip
import torch
from PIL import Image
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)

print(f"CLIP loaded on {device}!")

In [ ]:
def encode_image_style(img):
    """Takes a PIL image, returns its CLIP embedding"""
    image_tensor = clip_preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        embedding = clip_model.encode_image(image_tensor)
    return embedding.cpu()

# Test it on your existing result
embedding = encode_image_style(result_composite)
print(f"Style embedding shape: {embedding.shape}")
print("Encoder working!")

In [ ]:
# This simulates a user liking and disliking certain styles
liked_images = []
disliked_images = []

def like_image(img):
    liked_images.append(encode_image_style(img))
    print(f"👍 Liked! Total liked: {len(liked_images)}")

def dislike_image(img):
    disliked_images.append(encode_image_style(img))
    print(f"👎 Disliked! Total disliked: {len(disliked_images)}")

def get_style_profile():
    """Builds user style profile from liked/disliked images"""
    if not liked_images:
        return None
    
    liked_mean = torch.stack(liked_images).mean(0)
    
    if disliked_images:
        disliked_mean = torch.stack(disliked_images).mean(0)
        # Pull toward liked, push away from disliked
        profile = liked_mean - (disliked_mean * 0.3)
    else:
        profile = liked_mean
    
    return profile

print("Preference system ready!")

In [ ]:
# Generate 4 different styles to react to
test_styles = [
    ("scandinavian", "modern scandinavian room, white walls, minimal, photorealistic"),
    ("industrial", "industrial loft, exposed brick, dark tones, photorealistic"),
    ("bohemian", "bohemian room, earth tones, plants, warm, photorealistic"),
    ("luxury", "luxury room, gold accents, velvet, chandeliers, photorealistic"),
]

generated_styles = {}

for name, prompt in test_styles:
    output = pipe(
        prompt=prompt,
        negative_prompt="ugly, blurry, cartoon, unrealistic",
        image=depth_map,
        num_inference_steps=20,
        guidance_scale=7.5
    ).images[0]
    generated_styles[name] = output
    print(f"Generated: {name}")

print("All styles ready!")

In [ ]:
# Simulate user preferences
# User likes clean minimal styles
like_image(generated_styles["scandinavian"])
like_image(generated_styles["bohemian"])

# User dislikes dark heavy styles  
dislike_image(generated_styles["industrial"])
dislike_image(generated_styles["luxury"])

# Build their style profile
profile = get_style_profile()
print(f"Style profile shape: {profile.shape}")
print("User taste profile built!")

In [ ]:
# Define style anchors here so everything below can use them
style_anchors = [
    "scandinavian minimalist interior",
    "industrial dark loft interior",
    "bohemian warm cozy interior",
    "luxury royal opulent interior",
    "modern clean contemporary interior",
    "rustic farmhouse interior",
    "japandi zen minimal interior"
]

# Generate 4 different styles to react to
test_styles = [
    ("scandinavian", "modern scandinavian room, white walls, minimal, photorealistic"),
    ("industrial", "industrial loft, exposed brick, dark tones, photorealistic"),
    ("bohemian", "bohemian room, earth tones, plants, warm, photorealistic"),
    ("luxury", "luxury room, gold accents, velvet, chandeliers, photorealistic"),
]

generated_styles = {}

for name, prompt in test_styles:
    output = pipe(
        prompt=prompt,
        negative_prompt="ugly, blurry, cartoon, unrealistic",
        image=depth_map,
        num_inference_steps=20,
        guidance_scale=7.5
    ).images[0]
    generated_styles[name] = output
    print(f"Generated: {name}")

print("All styles ready!")

In [ ]:
personalized_prompt = f"a {best_style}, \
                        photorealistic, \
                        interior design magazine, 8k, \
                        warm lighting"

personalized_result = pipe(
    prompt=personalized_prompt,
    negative_prompt="ugly, blurry, cartoon, unrealistic",
    image=depth_map,
    num_inference_steps=20,
    guidance_scale=7.5
).images[0]

print(f"Generated with personalized prompt: {personalized_prompt}")
personalized_result

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(image)
axes[0].set_title("Original Room", fontsize=13)
axes[0].axis("off")

axes[1].imshow(generated_styles["luxury"])
axes[1].set_title("Generic Suggestion\n(before taste learning)", fontsize=13)
axes[1].axis("off")

axes[2].imshow(personalized_result)
axes[2].set_title(f"Personalized Suggestion\n({best_style})", fontsize=13)
axes[2].axis("off")

plt.suptitle("Taste Learning in Action", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("taste_learning_demo.png")
plt.show()
print("Saved!")

In [ ]:
from PIL import Image, ImageFilter
import numpy as np

# Style anchors must be defined before full_pipeline
style_anchors = [
    "scandinavian minimalist interior",
    "industrial dark loft interior",
    "bohemian warm cozy interior",
    "luxury royal opulent interior",
    "modern clean contemporary interior",
    "rustic farmhouse interior",
    "japandi zen minimal interior"
]

def full_pipeline(input_image, user_id="default",
                  click_point=None, base_prompt="a beautiful room"):

    print(f"Processing for user: {user_id}")

    # Step 1 — Resize input
    input_image = input_image.resize((512, 512))

    # Step 2 — Get depth map
    depth_result = depth_estimator(input_image)
    depth_map = depth_result["depth"].resize((512, 512))
    print("✅ Depth map generated")

    # Step 3 — Get style modifier from user profile
    if liked_images:
        profile = get_style_profile()
        profile_flat = profile.squeeze().float()
        text_tokens = clip.tokenize(style_anchors).to(device)
        with torch.no_grad():
            text_embeddings = clip_model.encode_text(
                               text_tokens).cpu().float()
        similarities = torch.cosine_similarity(
            profile_flat.unsqueeze(0).expand_as(text_embeddings),
            text_embeddings
        )
        style_modifier = style_anchors[similarities.argmax().item()]
        final_prompt = f"{base_prompt}, {style_modifier}, \
                        photorealistic, interior design magazine"
    else:
        final_prompt = f"{base_prompt}, photorealistic, \
                        interior design magazine"

    print(f"✅ Style prompt: {final_prompt}")

    # Step 4 — Optional object segmentation
    if click_point is not None:
        image_np = np.array(input_image)
        predictor.set_image(image_np)
        click_points = np.array([click_point])
        click_labels = np.array([1])
        masks, _, _ = predictor.predict(
            point_coords=click_points,
            point_labels=click_labels,
            multimask_output=False
        )
        mask_image = Image.fromarray(
                     (masks[0] * 255).astype(np.uint8))
        mask_blurred = mask_image.filter(
                       ImageFilter.GaussianBlur(radius=3))
        print("✅ Object segmented")
    else:
        mask_blurred = None
        print("✅ Full room mode (no object selected)")

    # Step 5 — Generate redesign
    result = pipe(
        prompt=final_prompt,
        negative_prompt="ugly, blurry, cartoon, unrealistic",
        image=depth_map,
        num_inference_steps=20,
        guidance_scale=7.5
    ).images[0]
    print("✅ Redesign generated")

    # Step 6 — Apply mask if object mode
    if mask_blurred is not None:
        result = Image.composite(result, input_image, mask_blurred)
        print("✅ Object composite applied")

    print("🎉 Pipeline complete!")
    return result

print("full_pipeline ready!")

In [ ]:
# Test 1 — Full room redesign with better prompt
print("=== TEST 1: Full Room ===")
result1 = full_pipeline(
    input_image=image,
    user_id="student_1",
    base_prompt="same room with white walls, \
                 light grey sofa, wooden floor, \
                 bright natural lighting, minimal decor, \
                 no wood panels, open airy space, \
                 scandinavian apartment"
)
result1
# Test 2 — Object only (separate cell)
print("=== TEST 2: Object Only ===")
result2 = full_pipeline(
    input_image=image,
    user_id="student_1",
    click_point=[256, 280],
    base_prompt="a modern sofa, clean fabric, \
                 scandinavian style, wooden legs"
)
result2

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(image)
axes[0].set_title("Original Room", fontsize=13)
axes[0].axis("off")

axes[1].imshow(result1)
axes[1].set_title("Full Room Redesign", fontsize=13)
axes[1].axis("off")

axes[2].imshow(result2)
axes[2].set_title("Object Level Redesign", fontsize=13)
axes[2].axis("off")

plt.suptitle("Full Pipeline — RoomAI", 
             fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("final_pipeline_demo.png")
plt.show()
print("✅ Saved as final_pipeline_demo.png")

In [ ]:
guidance_scales = [5.0, 7.5, 10.0, 12.0]
guidance_results = []

for gs in guidance_scales:
    output = pipe(
        prompt="a modern scandinavian living room, \
                warm lighting, minimalist furniture, \
                hardwood floors, photorealistic",
        negative_prompt="ugly, blurry, cartoon, unrealistic",
        image=depth_map,
        num_inference_steps=20,
        guidance_scale=gs
    ).images[0]
    guidance_results.append(output)
    print(f"Generated guidance_scale={gs}")

print("All done!")

In [ ]:
inference_steps = [10, 20, 30, 50]
steps_results = []

for steps in inference_steps:
    output = pipe(
        prompt="a modern scandinavian living room, \
                warm lighting, minimalist furniture, \
                hardwood floors, photorealistic",
        negative_prompt="ugly, blurry, cartoon, unrealistic",
        image=depth_map,
        num_inference_steps=steps,
        guidance_scale=7.5
    ).images[0]
    steps_results.append(output)
    print(f"Generated steps={steps}")

print("All done!")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for idx, (ax, result, gs) in enumerate(
        zip(axes, guidance_results, guidance_scales)):
    ax.imshow(result)
    ax.set_title(f"Guidance scale: {gs}", fontsize=12)
    ax.axis("off")

plt.suptitle("Effect of Guidance Scale on Output Quality",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("guidance_scale_comparison.png")
plt.show()
print("Saved!")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for idx, (ax, result, steps) in enumerate(
        zip(axes, steps_results, inference_steps)):
    ax.imshow(result)
    ax.set_title(f"Inference steps: {steps}", fontsize=12)
    ax.axis("off")

plt.suptitle("Effect of Inference Steps on Output Quality",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("inference_steps_comparison.png")
plt.show()
print("Saved!")